##Training

In [1]:
import numpy as np
import mediapipe as mp
from mediapipe.tasks.python import vision
from mediapipe.tasks import python as mp_python
from datasets import load_dataset
from tensorflow import keras
from sklearn.model_selection import train_test_split
from tqdm import tqdm

c:\Users\theke\Documents\Code\bharatanatyam-mudra-classification\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

HF_TOKEN = os.getenv('HF_TOKEN')
from huggingface_hub import login
login(HF_TOKEN)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:
from datasets import load_dataset

dataset = load_dataset("Samarth0710/bharatanatyam-mudra-dataset")
dataset = dataset.filter(lambda row: row["gesture_type"] == "single_hand")

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['image', 'label', 'gesture_type', 'label_id'],
        num_rows: 14827
    })
})


In [4]:
data = dataset["train"]

In [5]:
# Labels
labels = sorted(set(data["label"]))
label_to_id = {l: i for i, l in enumerate(labels)}

In [6]:
# MediaPipe
MODEL_PATH = "googletutorial/hand_landmarker.task"


In [7]:
base_options = mp_python.BaseOptions(model_asset_path=MODEL_PATH)
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE,
    num_hands=2
)
detector = vision.HandLandmarker.create_from_options(options)

In [8]:
def compute_finger_angles(pts):
    """Compute the angle at each finger joint using dot products"""
    # Finger joint indices: [base, mid, tip] for each finger
    fingers = [
        [1, 2, 3, 4],    # thumb
        [5, 6, 7, 8],    # index
        [9, 10, 11, 12], # middle
        [13, 14, 15, 16],# ring
        [17, 18, 19, 20] # pinky
    ]

    angles = []
    for finger in fingers:
        for i in range(len(finger) - 2):
            a = pts[finger[i]]
            b = pts[finger[i+1]]
            c = pts[finger[i+2]]

            ba = a - b
            bc = c - b

            cos_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-6)
            angle = np.arccos(np.clip(cos_angle, -1, 1))
            angles.append(angle)

    return np.array(angles)  # len = 10: 2 angles per finger


def compute_fingertip_distances(pts):
    """Distance from each fingertip to the wrist."""
    fingertips = [4, 8, 12, 16, 20]
    return np.array([np.linalg.norm(pts[i]) for i in fingertips])  # len = 5

In [9]:
def normalize_hand(hand_landmarks):
    pts = np.array([[lm.x, lm.y] for lm in hand_landmarks])

    # Normalize by translation
    pts -= pts[0]

    # Normalize by scale
    scale = np.linalg.norm(pts[9])
    if scale > 1e-6:
        pts /= scale

    # Normalize by rotation: align wrist->middle MCP vector to point straight up (0, -1)
    ref = pts[9]
    angle = np.arctan2(ref[0], -ref[1])

    cos_a, sin_a = np.cos(angle), np.sin(angle)
    R = np.array([
        [cos_a, -sin_a],
        [sin_a,  cos_a]
    ])

    pts = pts @ R.T

    # Use the normalized landmark data to get more features like finger angles and tip distances to wrist
    coords    = pts.flatten()
    angles    = compute_finger_angles(pts)
    distances = compute_fingertip_distances(pts)


    return np.concatenate([coords, angles, distances])

def extract_features(image):
    img = np.array(image.convert("RGB"))
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img)
    results = detector.detect(mp_image)

    if not results.hand_landmarks:
        return None
    feats = []

    for hand in results.hand_landmarks[:2]:
        feats.append(normalize_hand(hand))

    while len(feats) < 2:
        feats.append(np.zeros(57))

    return np.concatenate(feats)

In [10]:
# Build dataset
X, y = [], []

for sample in tqdm(data):
    feat = extract_features(sample["image"])
    if feat is None:
        continue
    X.append(feat)
    y.append(label_to_id[sample["label"]])

X = np.array(X)
y = np.array(y)

100%|██████████| 14827/14827 [03:50<00:00, 64.41it/s]


In [11]:
# Split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y
)


In [ ]:
# # Model
# model = keras.Sequential([
#     keras.layers.Input(shape=(114,)),
#     keras.layers.Dense(256, activation="relu"),
#     keras.layers.Dropout(0.2),
#     keras.layers.Dense(128, activation="relu"),
#     keras.layers.Dense(len(labels), activation="softmax")
# ])

# Best model\n",
model = keras.Sequential([
    keras.layers.Input(shape=(114,)),
    keras.layers.Dense(512, activation="relu"),
    keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(256, activation="relu"),
    keras.layers.Dense(len(labels), activation="softmax")
])
model.compile(
    optimizer=keras.optimizers.Adam(0.0003),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [13]:
# Train
model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=75)

Epoch 1/75


ValueError: Exception encountered when calling Sequential.call().

[1mInput 0 with name 'None' of layer 'dense' is incompatible with the layer: expected axis -1 of input shape to have value 84, but received input with shape (None, 114)[0m

Arguments received by Sequential.call():
  • inputs=tf.Tensor(shape=(None, 114), dtype=float32)
  • training=True
  • mask=None
  • kwargs=<class 'inspect._empty'>

In [ ]:
# Save
model.save("mudra_model.keras")

In [22]:
import json
with open("labels.json", "w") as f:
    json.dump(label_to_id, f)

In [19]:
import numpy as np
import mediapipe as mp
import json
import cv2
from PIL import Image
from tensorflow import keras
from mediapipe.tasks.python import vision
from mediapipe.tasks import python as mp_python
# from google.colab.patches import cv2_imshow

#load model
model = keras.models.load_model("mudra_model.keras")

with open("labels.json") as f:
    label_to_id = json.load(f)

id_to_label = {v: k for k, v in label_to_id.items()}

#mediapipe setup
# MODEL_PATH = "/content/hand_landmarker (3).task"

options = vision.HandLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=vision.RunningMode.IMAGE,
    num_hands=2
)

detector = vision.HandLandmarker.create_from_options(options)

#feature extraction
def extract_features(results):
    feats = []

    for hand in results.hand_landmarks[:2]:
        for lm in hand:
            feats.extend([lm.x, lm.y, lm.z])

    while len(feats) < 126:
        feats.extend([0]*63)

    return np.array(feats, dtype=np.float32)

#load the image
image_path = "googletutorial/handimage.jpg"

pil_image = Image.open(image_path)
frame = np.array(pil_image.convert("RGB"))

h, w, _ = frame.shape

mp_image = mp.Image(
    image_format=mp.ImageFormat.SRGB,
    data=frame
)

results = detector.detect(mp_image)

#predict and draw bounding box
if results.hand_landmarks:

    features = extract_features(results).reshape(1, -1)

    preds = model.predict(features, verbose=0)
    class_id = np.argmax(preds)
    confidence = np.max(preds)

    label_text = f"{id_to_label[class_id]} ({confidence:.2f})"

    for hand_landmarks in results.hand_landmarks:

        xs = [lm.x for lm in hand_landmarks]
        ys = [lm.y for lm in hand_landmarks]

        x_min = int(min(xs) * w)
        y_min = int(min(ys) * h)
        x_max = int(max(xs) * w)
        y_max = int(max(ys) * h)

        # Draw bounding box
        cv2.rectangle(frame, (x_min, y_min), (x_max, y_max), (0,255,0), 2)

        # Draw label
        cv2.putText(frame, label_text,
                    (x_min, y_min - 10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.8, (0,150,0), 2)

else:
    print("No hand detected")

scale = 0.5
resized_frame = cv2.resize(frame, None, fx=scale, fy=scale)

# Display
# from google.colab.patches import cv2_imshow
# cv2_imshow()
cv2.imshow('Landmarks', cv2.cvtColor(resized_frame, cv2.COLOR_RGB2BGR))
key = cv2.waitKey(0)